## infer

In [ ]:

import os

import sys
sys.path.append("/home/ubuntu/zs_cleaning/dune_codec")
from inference_utils_darya import *

from transformers import AutoTokenizer
tokenizer_path = "KavirLabs/Darya_Tokenizer_bpe"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, add_bos_token=True, add_eos_token=True)

device = 'cuda'


import sys
sys.path.append("dune_codec")



device = 'cuda'
from codec.audio_processing.dune_codec import load_dune_audio_tokenizer
dac_model = load_dune_audio_tokenizer("Respair/dune_codec", 
                                         device=device)


CONFIG_PATH = "config_transformer.json"
CKPT_DIR    = "exp/darya_fix/checkpoints/step_0080000"



tts_model, tts_cfg = load_model_from_checkpoint(CONFIG_PATH, CKPT_DIR)



duration_model = load_duration_model(
    "duration_predictor/dur/exp/ckpts_02/multilingual_ce_fixed/checkpoints/step_0075000.pt",
    vocab_size=4096,
    latent_dim=52,
    hidden_dim=256,
    n_text_layer=8,
    n_cross_layer=8,
    n_head=8,
    output_dim=378,
    device=device,
)


silence_pack = torch.load("silence_fsq.pt", map_location="cpu", weights_only=True)

silence_fsq = silence_pack["latents"]
silence_codec_rate_hz = silence_pack.get("codec_rate_hz", 12.5)

In [ ]:

import re
import time
import torch


text_list = ["<S1><en> write your input text here. <S2> uh, maybe another speaker perhaps."]
utterances = [text_list] if isinstance(text_list, str) else text_list

td = Extractor(tokenizer, tts_cfg, device, duration_model, speaker_model=None)


text_ids, text_mask, _, _ = td.get_tokens(
    utterances,
    utterances,
)


speed = 1.15

# duration computed per utterance
duration_items = []

for utterance in utterances:
    duration_text = strip_tags(utterance)

    _, _, duration_text_ids_i, duration_text_mask_i = td.get_tokens(
        utterance,
        duration_text,
    )

    duration_i = td.get_duration(
        duration_text_ids_i,
        duration_text_mask_i,
        speed=speed,
    )

    duration_items.append(duration_i.squeeze(0))

duration = torch.stack(duration_items, dim=0).to(device=device, dtype=torch.long)

n_frame_per_class = 1
max_duration = int(tts_cfg["data"]["max_audio_seconds"] * tts_cfg["data"]["codec_rate_hz"])

torch.cuda.synchronize()
t0 = time.perf_counter()
latents = sample_euler_reducio_spk(
    tts_model,
    text_ids,
    text_mask,
    duration=duration,
    cond_latents=None,
    cond_latent_mask=None,
    duration_model=duration_model,
    n_frame_per_class=n_frame_per_class,
    max_duration=max_duration,
    steps=16,
    cfg=3.,
    seed=None,
)

silence_seconds = 0.3
silence_frames = int(round(silence_seconds * 12.5))

sil = silence_fsq[:silence_frames].to(
    device=latents.device,
    dtype=latents.dtype,
)

latents_merged = torch.cat(
    [
        x
        for i in range(latents.shape[0])
        for x in (
            [latents[i, :int(duration[i].item())], sil]
            if i < latents.shape[0] - 1
            else [latents[i, :int(duration[i].item())]]
        )
    ],
    dim=0,
)

audio_cropped = decode_audio(
    dac_model,
    latents_merged.transpose(0, 1),
)
torch.cuda.synchronize()
total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf = total_time / max(audio_duration, 1e-6)

print("utterances:", utterances)
print("duration:", duration.tolist())
print("rtf:", rtf)

Sawt(audio_cropped, rate=44100, normalize=False)

utterances: ['<S1><en> Oh my god, <S2> 🤣 <S1> ... I am actually losing my mind right now! Holy shit bro 🤣 You have absolutely no idea how perfect this is. That is hands-down the most amazing thing I have ever seen in my life!']
duration: [163]
rtf: 0.1307852395703592


In [ ]:

Sawt(audio_cropped, rate=44100, normalize=False)

utterances: ['<S1><en> Oh my god 🤣 I am actually losing my mind right now! Holy shit bro 🤣 You have absolutely no idea how perfect this is. That is hands-down the most amazing thing I have ever seen in my life.']
duration: [163]
rtf: 0.12585385544532682


## prompt - infer

In [ ]:
AUDIO_PATH = "/home/ubuntu/angry_guy.mp3"
import librosa


wav = librosa.load(AUDIO_PATH, sr=22050)[0]

prompt_tensor = torch.from_numpy(wav)
# prompt_text = decode_ids(beam_text[0])
prompt_text = '<S1><en> prompt transcription here.'
print(prompt_text)



@torch.no_grad()
def extract_prequant_latents(dac_model, wav: torch.Tensor, device="cuda"):
    wav = wav.float()

    audio = wav.unsqueeze(0).to(device)
    audio_len = torch.tensor([wav.shape[0]], dtype=torch.long, device=device)

    encoder = dac_model.codec.audio_encoder
    pre_q, latent_len = encoder(audio=audio, audio_len=audio_len)
    
    pre_q = pre_q.transpose(1, 2)

    valid_len = int(latent_len[0].item())
    pre_q = pre_q[:, :valid_len, :]
    return pre_q[0]

prompt_lats = extract_prequant_latents(dac_model, prompt_tensor, device=device)


<S1><en> They were not! Their memory serves as an example to us all! The courageous fallen, the anguished fallen! Their lives have meaning because we, the living, refuse to forget them!
fixed pre_q: torch.Size([1, 169, 52])


In [ ]:
import time
import torch
import re

text = "Some text, tags should only appear in the prompt transcription and not here. unless it is cross lingual, otherwise the accent will leak."

td = Extractor(tokenizer, tts_cfg, device, duration_model, None)

full_text = prompt_text + "" + text
duration_text = re.sub(r"<[^>]+>\s*", "", full_text).strip()

text_ids, text_mask, duration_text_ids, duration_text_mask = td.get_tokens(
    full_text,
    duration_text,
)

prompt_lats_b = prompt_lats.unsqueeze(0)

prompt_mask = torch.ones(
    1,
    prompt_lats.shape[0],
    device=device,
    dtype=torch.bool,
)

n_frame_per_class = 1
max_duration = td.max_duration
speed = 1.

T_prompt = prompt_lats.shape[0]

duration = td.get_duration(
    duration_text_ids,
    duration_text_mask,
    speed=speed,
    cond_latents=prompt_lats_b,
    cond_latent_mask=prompt_mask,
)

torch.cuda.synchronize()
t0 = time.perf_counter()

output = sample_euler_reducio(
    tts_model,
    text_ids=text_ids,
    text_mask=text_mask,
    duration=duration,
    cond_latents=prompt_lats_b,
    cond_latent_mask=prompt_mask,
    duration_model=duration_model,
    n_frame_per_class=n_frame_per_class,
    max_duration=max_duration,
    steps=32,
    cfg=3.0,
    seed=None,
)

continuation_lats = output[:, T_prompt:int(duration[0].item()), :]

audio_cropped = decode_audio(
    dac_model,
    continuation_lats.transpose(1, 2),
)

torch.cuda.synchronize()
total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf = total_time / max(audio_duration, 1e-6)

print(
    f"used_frames={int(duration[0].item())} | "
    f"generated_frames={continuation_lats.shape[1]} | "
    f"Total: {total_time:.3f}s | Audio: {audio_duration:.2f}s | RTF: {rtf:.4f}"
)

Sawt(audio_cropped, rate=44100)

used_frames=324 | generated_frames=155 | Total: 3.074s | Audio: 12.40s | RTF: 0.2479


In [ ]:


Sawt(audio_cropped, rate=44100)

used_frames=212 | generated_frames=144 | Total: 3.506s | Audio: 11.52s | RTF: 0.3043


# Style Conditioned Path

Here you can prompt the model another way. <br> 
The benefits of this approach are :
- no need to load an ASR or provide prompt transcriptions
- context length of the model is preserved, because the prompt is a fixed global vector and does not eat the context away. 
- You can use cross-lingual prompts, regardless of whether the model has seen that language or not.
- mix various clips by doing a convex combination of their vectors, creating new voices.

the downside is that the model is limited to the capacity of Titanet, which is trained on a limited amount of English-only data. <br>
The checkpoint is also not trained and optimized from scratch with this approach, so it can impact the acoustic quality.

If you want the maximum stability / quality and did not like what you see below, feel free to go back to the infilling path.

## startup

In [ ]:

# ---- paths ----
CONFIG_PATH = "config_transformer.json"
CKPT_DIR    = "exp/darya_fix_2ndstage/checkpoints/step_0070000" # load the model trained with a speaker encoder

tts_model, tts_cfg = load_model_from_checkpoint(CONFIG_PATH, CKPT_DIR, use_speaker_conditioning=True)

import nemo.collections.asr as nemo_asr
speaker_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large")

## infer

### long form or simple

at least the way I am formulating it.

- NOTE: since Titanet speaker embeds are pooled global tensors, they were not made to handle multi speakers. try to keep it limited to single speaker inputs

In [ ]:
"""long form one"""

import re
import time
import torch

torch.cuda.synchronize()
t0 = time.perf_counter()


text_list = [

    "<S1><en> some text"
]

# normalize single string/list into utterance list
utterances = [text_list] if isinstance(text_list, str) else text_list

td = Extractor(tokenizer, tts_cfg, device, duration_model, speaker_model)

# main TTS tokens
text_ids, text_mask, _, _ = td.get_tokens(
    utterances,
    utterances,
)


speed = 1.

emb = td.get_speaker_embedding("/home/ubuntu/angry_guy.mp3")
# duration computed per utterance
duration_items = []

for utterance in utterances:
    duration_text = strip_tags(utterance)

    _, _, duration_text_ids_i, duration_text_mask_i = td.get_tokens(
        utterance,
        duration_text,
    )

    duration_i = td.get_duration(
        duration_text_ids_i,
        duration_text_mask_i,
        speed=speed,
    )

    duration_items.append(duration_i.squeeze(0))

duration = torch.stack(duration_items, dim=0).to(device=device, dtype=torch.long)

n_frame_per_class = 1
max_duration = int(tts_cfg["data"]["max_audio_seconds"] * tts_cfg["data"]["codec_rate_hz"])



In [ ]:

latents = sample_euler_reducio_spk(
    tts_model,
    text_ids,
    text_mask,
    duration=duration,
    cond_latents=None,
    cond_latent_mask=None,
    duration_model=duration_model,
    n_frame_per_class=n_frame_per_class,
    max_duration=max_duration,
    speaker_emb=emb,
    speaker_adaln_scale=2.,
    steps=32,
    cfg=2.,
    speaker_cfg=1.5, # less like the speaker
    seed=42,
)

silence_seconds = 0.3
silence_frames = int(round(silence_seconds * 12.5))

sil = silence_fsq[:silence_frames].to(
    device=latents.device,
    dtype=latents.dtype,
)

latents_merged = torch.cat(
    [
        x
        for i in range(latents.shape[0])
        for x in (
            [latents[i, :int(duration[i].item())], sil]
            if i < latents.shape[0] - 1
            else [latents[i, :int(duration[i].item())]]
        )
    ],
    dim=0,
)

audio_cropped = decode_audio(
    dac_model,
    latents_merged.transpose(0, 1),
)
torch.cuda.synchronize()
total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf = total_time / max(audio_duration, 1e-6)

print("utterances:", utterances)
print("duration:", duration.tolist())
print("rtf:", rtf)

Sawt(audio_cropped, rate=44100, normalize=False)

### long form sequential

each generation is conditioned on the one that came before it.
in production, each chunk can and should be played after it is decoded.



In [ ]:
import re
import time
import torch
import numpy as np

torch.cuda.synchronize()
t0 = time.perf_counter()

# emb0 = td.get_speaker_embedding(audio)
emb0 = None
emb = emb0

audio_chunks = []

for i, utterance in enumerate(text_list):
    duration_text = strip_tags(utterance)

    text_ids, text_mask, duration_text_ids, duration_text_mask = td.get_tokens(
        utterance,
        duration_text,
    )

    duration = td.get_duration(
        duration_text_ids,
        duration_text_mask,
        speed=1.,
    )

    latents = sample_euler_reducio_spk(
        tts_model,
        text_ids,
        text_mask,
        duration=duration,
        cond_latents=None,
        cond_latent_mask=None,
        duration_model=duration_model,
        n_frame_per_class=n_frame_per_class,
        max_duration=max_duration,
        speaker_emb=emb,
        speaker_adaln_scale=1.,
        steps=32,
        cfg=3.,
        speaker_cfg=None,
        seed=None,
    )

    dur_i = int(duration[0].item())
    latents_i = latents[0, :dur_i]

    audio_cropped = decode_audio(
        dac_model,
        latents_i.transpose(0, 1),
    )

    if isinstance(audio_cropped, torch.Tensor):
        audio_np = audio_cropped.detach().float().cpu().squeeze().numpy()
    else:
        audio_np = np.asarray(audio_cropped, dtype=np.float32).squeeze()

    audio_chunks.append(audio_np)

    emb_prev = td.get_speaker_embedding(audio_np)

    if emb0 is None:
        emb0 = emb_prev
        emb = emb_prev
    else:
        emb = 0.25 * emb0 + 0.75 * emb_prev

audio_cropped = np.concatenate(audio_chunks, axis=-1)

torch.cuda.synchronize()
total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf = total_time / max(audio_duration, 1e-6)

print("text_list:", text_list)
print("rtf:", rtf)

Sawt(audio_cropped, rate=44100, normalize=False)

# SPEECH EDIT

In [ ]:
AUDIO_TO_EDIT =  "/mnt/tradeoffs.mp3"

target_text = '<S1><en> please put your target text here with the corrected segments.'

wav, wav_sr = librosa.load(AUDIO_TO_EDIT, sr=22050)
wav_tensor = torch.from_numpy(wav)

original_lats = extract_prequant_latents(dac_model, wav_tensor, device=device)

codec_rate_hz = tts_cfg["data"]["codec_rate_hz"]

parts_to_edit = [
    [0, 2],
    [7, 9],     
]

extra_duration = [
    0.5,   # add room 
]

cond, keep_mask, span_mask, valid, edit_ranges = build_latent_edit_condition(
    original_lats,
    wav_num_samples=len(wav),
    wav_sr=wav_sr,
    parts_to_edit=parts_to_edit,
    extra_duration=extra_duration,
    duration_scale=1.0,
    padding_sec=0.5,
    min_edit_sec=0.15,
    codec_rate_hz=codec_rate_hz,
    device=device,
    dtype=tts_model.dtype,
)

td = Extractor(tokenizer, tts_cfg, device, duration_model, None)

duration_text = re.sub(r"<[^>]+>\s*", "", target_text).strip()

text_ids, text_mask, _, _ = td.get_tokens(
    target_text,
    duration_text,
)

edited_lats = sample_euler_reducio_edit(
    tts_model,
    text_ids=text_ids,
    text_mask=text_mask,
    cond=cond,
    keep_mask=keep_mask,
    valid_mask=valid,
    steps=32,
    cfg=2.,
    seed=None,
)

audio_edited = decode_audio(
    dac_model,
    edited_lats.transpose(1, 2),
)

Sawt(audio_edited, rate=44100)